In [8]:
import websockets
import polars as pl
import json
import httpx
import json
import time
import random
from pathlib import Path
from collections import Counter
import parquet
import asyncio
from datetime import datetime, timezone

In [2]:
#Movie/tv show search terms
WATCHED_SEARCH_QUERIES = [
    "just watched",
    "just finished watching",
    "finally watched",
    "finally saw",
    "just saw",
    "movie was",
    "film was",
    "watched last night",
    "binge watched",
    "finished the series",
    "finished the season",
    "finished the movie",
    "great movie",
    "terrible movie",
    "loved the movie",
    "hated the movie",
    "great film",
    "terrible film",
    "hated the film",
    "loved the film",
    "movie",
    "film",
    "cinema",
    "rewatching",
]

In [3]:
#Man United search terms - barely any live hits, avoid this for now
FOOTBALL_SEARCH_QUERIES = [
    "Manchester United",
    "Man United",
    "Man U",
    "Manchester Utd",
    "Red Devils",
    "Mbeumo",
    "Dalot",
    "Diogo Dalot",
    "Rashford",
    "Ineos",
    "Cunha",
    "Sesko",
    # "UCL",
    # "Champions League",
    # "Premier League",
    # "EPL",
    # "UEFA Champions League",
]

In [4]:
#Pokemon search terms - barely any live hits, avoid this for now
POKEMON_SEARCH_QUERIES = [
    "Pokemon",
    "Pikachu",
    "Charizard",
    "Bulbasaur",
    "Squirtle",
    "Eevee",
    "Mewtwo",
    "Gengar",
    "Snorlax",
    "Jigglypuff",
    "Ascended Heroes",
    "Prismatic Evolutions",
    "Destined Rivals",
]

In [ ]:
#Code adapted from Jetstream docs https://bsky.network/docs/jetstream/

#connect to the live jetstream, collecting posts until MAX_POSTS is hit
MAX_POSTS = 20000
rows = []
total_scanned = 0
#add a timeout to prevent searching indefinitely
timeout = 300
timed_out = False
start_time = time.time()

#performing server-side filtering using "&kinds=commit" to only receive posts

#(note that this plays "live" but very sped up due to async, so lets you process all the posts over the timescale provided)
#replay the last 3 hours
REPLAY_HOURS = 3
#init a cursor to track the last post received
last_cursor = int((time.time() - REPLAY_HOURS * 3600) * 1_000_000)

WS_URI = (
    f"wss://jetstream.us-east.bsky.network/subscribe"
    f"?wantedCollections=app.bsky.feed.post"
    f"&kinds=commit"
)

consecutive_live = 0

#jetstream cuts off connection during high traffic or high throughput, so using cursor to pick
#back up where connection drops. This gets cut off at MAX_POSTS to enforce an upper limit on posts processed
#or when timeout is reached, or when all posts from past 3 hours have been collected

while len(rows) < MAX_POSTS and not timed_out:
    uri = f"{WS_URI}&cursor={last_cursor}"
    try:
        async with websockets.connect(uri) as ws:
            try:
                async for frame in ws:
                    event = json.loads(frame)

                    #filter server-side for posts only, skip other events
                    rec = event.get("commit", {}).get("record")
                    if not rec:
                        continue

                    #update cursor
                    last_cursor = event.get("cursor", last_cursor)

                    # detect when caught up to live by detecting 20 consecutive posts which are all within 30 seconds of live.
                    #had to use 20 consecutive because some bluesky clients report incorrect created_at timestamps
                    created = datetime.fromisoformat(rec.get("createdAt").replace("Z", "+00:00"))
                    age_seconds = (datetime.now(timezone.utc) - created).total_seconds()
                    if age_seconds < 30:
                        consecutive_live += 1
                        if consecutive_live >= 20:
                            print(f"caught up to live, ending Jetstream connection")
                            timed_out = True
                            break
                    else:
                        consecutive_live = 0

                    #add timeout to avoid hanging indefinitely
                    if time.time() - start_time > timeout:
                        print(f"Timeout reached after {timeout} seconds.")
                        timed_out = True
                        break
            
                    #filter on english posts only
                    langs = rec.get("langs", [])
                    if "en" not in langs:
                        continue
                    
                    total_scanned += 1

                    text = rec.get("text", "")
                    text_lower = text.lower()        

                    #skip if no match on search terms
                    if not any(kw.lower() in text_lower for kw in WATCHED_SEARCH_QUERIES):
                        continue
            
                    rows.append({
                        "did": event.get("did", "unknown"),
                        "cursor": event.get("cursor"),
                        "created_at": rec.get("createdAt"),
                        "text": text,
                        "langs": str(rec.get("langs", [])),
                        "is_reply": rec.get("reply") is not None,
                        "has_embed": rec.get("embed") is not None,
                        "matched_keyword": next(kw for kw in WATCHED_SEARCH_QUERIES if kw.lower() in text_lower),
                    })

                    #prints output every 50 matches
                    if len(rows) % 50 == 0:
                        print(f"Matched {len(rows)} / scanned {total_scanned}")

                    #break when MAX_POSTS hit
                    if len(rows) >= MAX_POSTS:
                        break

            except websockets.ConnectionClosedError:
                    print(f"Connection has dropped; collected {len(rows)} matches from {total_scanned} total scanned.")
    except websockets.ConnectionClosedError:
        print(f"Reconnecting from cursor {last_cursor}...")
        await asyncio.sleep(2)
        continue
 
print(f"\nDone. {len(rows)} matches from {total_scanned} scanned posts.")

Matched 50 / scanned 4359
Matched 100 / scanned 9348
Matched 150 / scanned 13655
Matched 200 / scanned 17632
Matched 250 / scanned 22032
Matched 300 / scanned 25219
Matched 350 / scanned 28474
Matched 400 / scanned 34627
Matched 450 / scanned 38640
Matched 500 / scanned 42173
Matched 550 / scanned 45966
Matched 600 / scanned 51082
Matched 650 / scanned 54752
Matched 700 / scanned 58886
Matched 750 / scanned 63949
Matched 800 / scanned 68446
Matched 850 / scanned 73270
Matched 900 / scanned 78106
Matched 950 / scanned 82757
Matched 1000 / scanned 87488
Matched 1050 / scanned 91663
Matched 1100 / scanned 97035
Matched 1150 / scanned 101262
Matched 1200 / scanned 108281
Matched 1250 / scanned 112564
Matched 1300 / scanned 116918
Matched 1350 / scanned 122119
Matched 1400 / scanned 125872
Caught up to live, ending Jetstream connection

Done. 1430 matches from 129441 scanned posts.


In [9]:
df = pl.DataFrame(rows)
df.write_parquet("../data/backfill_watched.parquet")
df.write_csv("../data/backfill_watched.csv")
print(df.shape)
df.head(10)

(2492, 8)


did,cursor,created_at,text,langs,is_reply,has_embed,matched_keyword
str,i64,str,str,str,bool,bool,str
"""did:plc:zjpgmfkvug27pdtgxl7npn…",25241411161,"""2026-08-29T11:49:30.444Z""","""I love how he looks in the cin…","""['en']""",true,false,"""cinema"""
"""did:plc:23e6qmug2vwcnsfv2fywuy…",25241412907,"""2026-08-29T11:49:34.637Z""","""Me, a text: I am running late …","""['en']""",false,false,"""cinema"""
"""did:plc:6q5ygzzha3dkmgqpfatp5r…",25241413817,"""2026-08-29T11:49:40.178299Z""","""Radio C - Cinema e Cultura: Be…","""['en']""",false,false,"""cinema"""
"""did:plc:i7i76dfe63j2g3lkl43prl…",25241414765,"""2026-08-29T11:49:45.855Z""","""Make your favourite film sound…","""['en']""",false,true,"""film"""
"""did:plc:orhnxxrkuhlet5kcihd6cp…",25241415642,"""2026-08-29T11:49:47.074Z""","""Make your favorite film sound …","""['en']""",false,true,"""film"""
"""did:plc:3wycn25ebep6ptj3fajqa6…",25241418592,"""2026-08-29T11:49:57.325Z""","""Make your favorite film sound …","""['en']""",false,true,"""film"""
"""did:plc:wxajozzeaa677lcy6ju6cw…",25241418624,"""2026-08-29T11:49:10.528Z""","""Happy Saturday... This is #Man…","""['en']""",false,true,"""film"""
"""did:plc:idib4pjj4oylwswpavkxs7…",25241418908,"""2026-08-29T11:49:58.136Z""","""REALLY scary movie!!!""","""['en']""",false,true,"""movie"""
"""did:plc:lkcw5mcnwb2phdkh6pbqpk…",25241419438,"""2026-08-29T11:50:00.495Z""","""Make your favourite film sound…","""['en']""",false,true,"""film"""
